In [1]:
import cv2
import numpy as np
import matplotlib.pyplot as plt
imgf='straight.png'
omg=cv2.imread(imgf)

In [2]:
def disply(im_path):

    dpi = 80
    im_data = plt.imread(im_path)
    height, width, depth = im_data.shape

    # What size does the figure need to be in inches to fit the image?
    figsize = width / float(dpi), height / float(dpi)

    # Create a figure of the right size with one axes that takes up the full figure
    fig = plt.figure(figsize=figsize)
    ax = fig.add_axes([0, 0, 1, 1])

    # Hide spines, ticks, etc.
    ax.axis('off')

    # Display the image.
    ax.imshow(im_data, cmap='gray')

    plt.show()

In [3]:
# disply(imgf)
print(imgf)

straight.png


Inverted Image

In [4]:
#invertImg=cv2.bitwise_not(omg)
#cv2.imwrite('inverted.jpg',invertImg)
#disply('inverted.jpg')

Rescalling

GrayOut

In [5]:
def grayscale(image):
    return cv2.cvtColor(image, cv2.COLOR_BGR2GRAY)

In [6]:
graay=grayscale(omg)
cv2.imwrite('grayed.jpg',graay)
#plt.imshow('/home/mukesh/Project/PreproImg/temp/grayed.jpg')
# disply('temp/grayed.jpg')

True

Binarization

In [7]:
def binry(image,thres):
    thres, bwi=cv2.threshold(image,thres,300,cv2.THRESH_BINARY)
    return bwi

In [8]:
bthres=200
bblck=binry(graay,bthres)
cv2.imwrite('blackwhite.jpg',bblck)

True


Noise Removal 

In [9]:
# def rnoise(image):
#     kernaal=np.ones((2,2),np.uint8)
#     image=cv2.dilate(image,kernaal,iterations=1)
#     image=cv2.erode(image,kernaal,iterations=1)
#     image=cv2.morphologyEx(image,cv2.MORPH_CLOSE,kernaal)
#     image=cv2.medianBlur(image,3)
#     return image


In [ ]:
# nonois=rnoise(bblck)
# cv2.imwrite('nonoise.jpg', nonois)

Thinn (Erosion)

In [ ]:
# def thinf(image):
#     image=cv2.bitwise_not(image)
#     kernaal=np.ones((4,4),np.uint8)
#     image=cv2.erode(image,kernaal,iterations=1)
#     image=cv2.bitwise_not(image)
#     return image

In [ ]:
# thinn=thinf(nonois)
# cv2.imwrite('thined.jpg',thinn)

Thicken (Dilate)

In [ ]:
# def thickf(image):
#     image=cv2.bitwise_not(image)
#     kernaal=np.ones((2,2),np.uint8)
#     image=cv2.dilate(image,kernaal,iterations=1)
#     image=cv2.bitwise_not(image)
#     return image

In [ ]:
# thikk=thickf(nonois)
# cv2.imwrite('thicked.jpg',thikk)

Deskew (Rotaion)

In [ ]:
def getSkewAngle(cvImage) -> float:
    # Prep image, copy, convert to gray scale, blur, and threshold
    newImage = cvImage.copy()
    gray = cv2.cvtColor(newImage, cv2.COLOR_BGR2GRAY)
    blur = cv2.GaussianBlur(gray, (9, 9), 0)
    thresh = cv2.threshold(blur, 0, 255, cv2.THRESH_BINARY_INV + cv2.THRESH_OTSU)[1]

    # Apply dilate to merge text into meaningful lines/paragraphs.
    # Use larger kernel on X axis to merge characters into single line, cancelling out any spaces.
    # But use smaller kernel on Y axis to separate between different blocks of text
    kernel = cv2.getStructuringElement(cv2.MORPH_RECT, (30, 5))
    dilate = cv2.dilate(thresh, kernel, iterations=5)

    # Find all contours
    contours, hierarchy = cv2.findContours(dilate, cv2.RETR_LIST, cv2.CHAIN_APPROX_SIMPLE)
    contours = sorted(contours, key = cv2.contourArea, reverse = True)

    # Find largest contour and surround in min area box
    largestContour = contours[0]
    minAreaRect = cv2.minAreaRect(largestContour)

    # Determine the angle. Convert it to the value that was originally used to obtain skewed image
    angle = minAreaRect[-1]
    if angle < -45:
        angle = 90 + angle
    return -1.0 * angle

def rotateImage(cvImage, angle: float):
    newImage = cvImage.copy()
    (h, w) = newImage.shape[:2]
    center = (w // 2, h // 2)
    M = cv2.getRotationMatrix2D(center, angle, 1.0)
    newImage = cv2.warpAffine(newImage, M, (w, h), flags=cv2.INTER_CUBIC, borderMode=cv2.BORDER_REPLICATE)
    return newImage

def deskew(cvImage):
    angle = getSkewAngle(cvImage)
    return rotateImage(cvImage, -1.0 * angle)